# P3-3 · Language model và sinh văn bản

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
V = 50                                                   # kích thước từ vựng (nhỏ để chạy thử)
tokens = torch.randint(0, V, (2, 8))                     # 2 chuỗi, mỗi chuỗi 8 token
model = nn.Sequential(nn.Embedding(V, 32), nn.Linear(32, V))   # "model" tối giản, chưa train

x = tokens[:, :-1]                     # (B, T)   đầu vào
y = tokens[:, 1:]                      # (B, T)   đáp án: dịch trái 1 vị trí
logits = model(x)                      # (B, T, V)
loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
print(x.shape, y.shape, logits.shape)
print(f"loss {loss.item():.2f}, model đoán bừa thì khoảng ln V = {math.log(V):.2f}")

torch.Size([2, 7]) torch.Size([2, 7]) torch.Size([2, 7, 50])
loss 3.97, model đoán bừa thì khoảng ln V = 3.91


In [ ]:
@torch.no_grad()
def sample_next(logits, temperature=0.8, top_k=50):
    if temperature == 0:
        return logits.argmax(-1, keepdim=True)                # greedy
    logits = logits / temperature
    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
    logits[logits < v[..., -1, None]] = float("-inf")         # bỏ mọi token ngoài top-k
    return torch.multinomial(torch.softmax(logits, -1), 1)

# thử với 5 token có logit khác nhau: đếm token nào được chọn trong 1.000 lần
torch.manual_seed(0)
logits = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0])
for t, k in [(0, 50), (0.5, 50), (1.0, 50), (1.0, 2)]:
    picks = torch.cat([sample_next(logits.clone(), t, k) for _ in range(1000)])
    print(f"temperature {t}, top_k {k}:", torch.bincount(picks, minlength=5).tolist())

temperature 0, top_k 50: [1000, 0, 0, 0, 0]
temperature 0.5, top_k 50: [874, 107, 16, 3, 0]
temperature 1.0, top_k 50: [666, 204, 87, 36, 7]
temperature 1.0, top_k 2: [735, 265, 0, 0, 0]


## Bài tập 5: Viết hàm sinh với top-p {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch

def sample_top_p(logits, p=0.9, temperature=1.0):
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_p, idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    keep = cum - sorted_p < p          # giữ token chừng nào tổng trước nó còn < p
    sorted_p = sorted_p * keep
    choice = torch.multinomial(sorted_p / sorted_p.sum(), 1)
    return idx[choice].item()

print(sample_top_p(torch.tensor([2.0, 1.0, 0.0, -1.0]), p=0.9))

1
